# Lab 22 — nested LOCO hyperparameter tuning with Optuna

Mục tiêu: kiểm tra Hyperparameter Tuning có cải thiện robustness multicenter hay không.

- Dataset: 920 real rows, bốn UCI hospitals.
- Validation: outer Leave-One-Center-Out; inner GroupKFold theo hospital.
- Preprocessing: P1 sentinel-aware, giữ đúng pipeline Lab 17/18.
- Models: Logistic Regression và LightGBM.
- So sánh fixed baseline với Optuna-tuned model.
- Threshold giữ `0.50`; constrained threshold của Lab 21 được đánh giá sau khi model selection hoàn tất.

Optuna không được nhìn outer test hospital.

In [ ]:
!pip -q install optuna lightgbm seaborn

import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
N_TRIALS = 30
THRESHOLD = 0.50
OUTPUT_DIR = Path('/content/uci_multicenter_nested_optuna_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
COLUMNS = FEATURES + ['num']

def read_uci(site, filename):
    frame = pd.read_csv(f'{BASE_URL}/{filename}', names=COLUMNS, na_values=['?'],
                        skipinitialspace=True).apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
SITES = list(FILES.keys())
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))

## 1. P1 preprocessing và model builders

Feature Engineering không được mở rộng trong Lab 22 để tách riêng tác động của hyperparameters.

In [ ]:
FIXED_PARAMS = {
    'Logistic Regression': {'max_iter': 2000},
    'LightGBM': {'n_estimators': 250, 'learning_rate': 0.03, 'num_leaves': 15,
        'min_child_samples': 15, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_lambda': 1.0},
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def build_pipeline(model_name, params=None):
    params = dict(params or {})
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer([('numeric', numeric, NUMERICAL_FEATURES),
                                     ('categorical', categorical, CATEGORICAL_FEATURES)])
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(random_state=RANDOM_STATE, **params)
    else:
        estimator = LGBMClassifier(random_state=RANDOM_STATE, verbosity=-1, **params)
    return Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])

def prepare(frame):
    ready = apply_p1(frame)
    return ready[FEATURES], ready[TARGET]

def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'false_positives': int(fp)}


## 2. Optuna chỉ tune trong inner GroupKFold

Mỗi trial được chấm bằng mean inner ROC-AUC trên ba training hospitals. Không dùng outer test hospital để chọn params.

In [ ]:
def suggest_params(trial, model_name):
    if model_name == 'Logistic Regression':
        return {
            'C': trial.suggest_float('C', 0.01, 10.0, log=True),
            'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
            'max_iter': 2000,
        }
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 7, 63),
        'max_depth': trial.suggest_categorical('max_depth', [-1, 3, 5, 8]),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.70, 1.00),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.70, 1.00),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
    }

def tune_on_training(train_frame, model_name):
    y = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    splits = list(GroupKFold(n_splits=3).split(train_frame, y, groups))

    def objective(trial):
        params = suggest_params(trial, model_name)
        fold_scores = []
        for inner_train_idx, inner_valid_idx in splits:
            inner_train = train_frame.iloc[inner_train_idx]
            inner_valid = train_frame.iloc[inner_valid_idx]
            X_fit, y_fit = prepare(inner_train)
            X_valid, y_valid = prepare(inner_valid)
            model = build_pipeline(model_name, params)
            model.fit(X_fit, y_fit)
            probability = model.predict_proba(X_valid)[:, 1]
            fold_scores.append(roc_auc_score(y_valid, probability))
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
    started = time.perf_counter()
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=False)
    tuning_seconds = time.perf_counter() - started
    return study, tuning_seconds


In [ ]:
outer_records = []
best_param_records = []
trial_records = []
for test_site in SITES:
    train_frame = data[data['site'] != test_site].reset_index(drop=True)
    test_frame = data[data['site'] == test_site].reset_index(drop=True)
    X_train, y_train = prepare(train_frame)
    X_test, y_test = prepare(test_frame)
    for model_name in ['Logistic Regression', 'LightGBM']:
        fixed_model = build_pipeline(model_name, FIXED_PARAMS[model_name])
        started = time.perf_counter(); fixed_model.fit(X_train, y_train); fixed_fit_seconds = time.perf_counter() - started
        fixed_probability = fixed_model.predict_proba(X_test)[:, 1]
        fixed_metrics = score_probability(y_test, fixed_probability)
        outer_records.append({'configuration': 'F0_P1_fixed', 'model': model_name,
            'test_site': test_site, 'tuning_seconds': 0.0, 'fit_seconds': fixed_fit_seconds,
            'best_inner_roc_auc': np.nan, 'best_params': '{}', **fixed_metrics})

        study, tuning_seconds = tune_on_training(train_frame, model_name)
        best_params = study.best_trial.params.copy()
        if model_name == 'Logistic Regression': best_params['max_iter'] = 2000
        best_param_records.append({'test_site': test_site, 'model': model_name,
            'best_inner_roc_auc': study.best_value, 'best_params': json.dumps(best_params, sort_keys=True),
            'tuning_seconds': tuning_seconds, 'n_trials': len(study.trials)})
        for trial in study.trials:
            trial_records.append({'test_site': test_site, 'model': model_name,
                'trial_number': trial.number, 'state': str(trial.state),
                'value': trial.value, 'params': json.dumps(trial.params, sort_keys=True)})

        tuned_model = build_pipeline(model_name, best_params)
        started = time.perf_counter(); tuned_model.fit(X_train, y_train); fit_seconds = time.perf_counter() - started
        tuned_probability = tuned_model.predict_proba(X_test)[:, 1]
        tuned_metrics = score_probability(y_test, tuned_probability)
        outer_records.append({'configuration': 'F0_P1_optuna_nested', 'model': model_name,
            'test_site': test_site, 'tuning_seconds': tuning_seconds, 'fit_seconds': fit_seconds,
            'best_inner_roc_auc': study.best_value, 'best_params': json.dumps(best_params, sort_keys=True),
            **tuned_metrics})
    print('Completed outer test site:', test_site)

results_df = pd.DataFrame(outer_records)
best_params_df = pd.DataFrame(best_param_records)
trials_df = pd.DataFrame(trial_records)
display(results_df.round(4))
display(best_params_df)

## 3. Tổng hợp Optuna vs fixed baseline

Threshold ở đây vẫn là 0.50. Sau khi chọn tuned model, có thể đưa đúng model đó qua Lab 21 để chọn constrained operating point.

In [ ]:
summary = results_df.groupby(['configuration', 'model']).agg(
    folds=('test_site', 'nunique'), roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'), roc_auc_worst=('roc_auc', 'min'),
    pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'), recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'), f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'), false_negatives_mean_per_fold=('false_negatives', 'mean'),
    false_negatives_total=('false_negatives', 'sum'), false_positives_total=('false_positives', 'sum'),
    best_inner_roc_auc_mean=('best_inner_roc_auc', 'mean'),
    tuning_seconds_mean=('tuning_seconds', 'mean'), fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

baseline = summary[summary['configuration'] == 'F0_P1_fixed'].set_index('model')
delta = summary.copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'pr_auc_mean', 'recall_mean',
                'recall_worst', 'brier_mean', 'false_negatives_total']:
    delta[f'delta_vs_fixed_{metric}'] = delta.apply(
        lambda row: row[metric] - baseline.loc[row['model'], metric], axis=1)

display(summary.sort_values(['model', 'roc_auc_worst'], ascending=[True, False]).round(6))
display(delta.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=summary, x='model', y='roc_auc_worst', hue='configuration', ax=axes[0])
axes[0].set_title('Worst-site ROC-AUC: fixed vs Optuna')
sns.barplot(data=summary, x='model', y='recall_mean', hue='configuration', ax=axes[1])
axes[1].set_title('Mean recall at threshold 0.50')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'nested_optuna_summary.png', dpi=180, bbox_inches='tight'); plt.show()

results_path = OUTPUT_DIR / 'nested_optuna_loco_results.csv'
params_path = OUTPUT_DIR / 'best_params_by_outer_fold.csv'
trials_path = OUTPUT_DIR / 'optuna_trial_history.csv'
summary_path = OUTPUT_DIR / 'nested_optuna_summary.csv'
delta_path = OUTPUT_DIR / 'nested_optuna_delta_vs_fixed.csv'
results_df.to_csv(results_path, index=False)
best_params_df.to_csv(params_path, index=False)
trials_df.to_csv(trials_path, index=False)
summary.to_csv(summary_path, index=False)
delta.to_csv(delta_path, index=False)
run_config = {'dataset_rows': 920, 'validation': 'outer LOCO + inner GroupKFold by hospital',
    'preprocessing': 'P1_sentinel_aware', 'threshold': THRESHOLD,
    'n_trials_per_model_per_outer_fold': N_TRIALS, 'optimization_metric': 'inner mean ROC-AUC',
    'tuned_models': ['Logistic Regression', 'LightGBM'], 'feature_engineering': 'none beyond P1',
    'class_weight': 'not searched', 'threshold_tuning': 'deferred to Lab 21',
    'outer_test_policy': 'real held-out hospital only'}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_nested_optuna_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)